# Classificatie van nieuwsartikelen

In deze notebook gaan we verder werken op de AG-news nieuwsartikelen dataset.
In de vorige notebook hebben we bekeken hoe we tekstuele data kunnen preprocessen.
In deze notebook gaan we classificatie uitvoeren door gebruik te maken van recurrente neurale netwerken.

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from collections import Counter
import re
import kagglehub

# Parameters
MAX_NUM_WORDS = 20000  # Maximum number of unique words to keep
MAX_SEQUENCE_LENGTH = 100  # Maximum length of input sequences
EMBEDDING_DIM = 50  # Dimensionality of the embedding layer

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device='cpu'
print(f"Using device: {device}")

# get the dataset
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

# Load the dataset
def read_csv(filename):
    df = pd.read_csv(filename)
    df.columns = ["label", "title", "description"]
    df["text"] = df['title'] + ' ' + df['description']
    df['label'] = df['label'] - 1
    return df

df_train = read_csv(f'{path}/train.csv')
df_test = read_csv(f'{path}/test.csv')
display(df_train.head())

# tokenizer
def simple_tokenizer(text):
    return re.findall(r"\b\w+\b", text.lower())
    
# Build vocab from training set
counter = Counter()
for text in df_train["text"]:
    counter.update(simple_tokenizer(text)) # splits de tekst kolom in woorden, tel hoeveel keer elk woord voorkomt

vocab = {word: idx+2 for idx, (word, _) in enumerate(counter.most_common(MAX_NUM_WORDS))}
vocab["<PAD>"] = 0 # special token for padding
vocab["<UNK>"] = 1 # special token for unknown words

print("Size of vocab:", len(vocab))

def encode(text, vocab, max_len=MAX_SEQUENCE_LENGTH):
    tokens = simple_tokenizer(text) # splits text in woorden
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens] # zet elk woord om naar een token (unknown als het niet bestaat)
    ids = ids[:max_len] # truncate de text
    ids += [vocab["<PAD>"]] * (max_len - len(ids)) # voeg padding toe als de tekst te kort is
    return ids

num_samples = 30000
X_train = [encode(text, vocab) for text in df_train["description"][:num_samples]]  # neem subset voor demo
y_train = df_train["label"][:num_samples]
X_train = torch.tensor(X_train)
print("X_train shape:", X_train.shape)
print('Highest token idx train', torch.max(X_train))

X_test = [encode(text, vocab) for text in df_test["description"][:num_samples]]  # neem subset voor demo
X_test = torch.tensor(X_test)
y_test = df_test["label"][:num_samples]
print('Highest token idx test', torch.max(X_test))

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Opbouwen, trainen en evalueren van een RNN

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(RNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        #print(x.shape, torch.max(x))
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded) # 1 laag met rnn-cellen (woord per woord wordt hierdoor gestuurd)
        last_output = hidden[-1]
        out = self.fc(last_output)
        return out

# Parameters
hidden_dim = 128
output_dim = 4  # AG-News heeft 4 klassen

model = RNNModel(MAX_NUM_WORDS+2, EMBEDDING_DIM, hidden_dim, output_dim)
print(model)

texts, labels = next(iter(train_loader))
print(texts.shape, labels.shape)
print(model(texts).shape)

In [ ]:

# Definieer de Loss en Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

# Train het Model
num_epochs = 2
model.train()
print("Training started.")

for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        #inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training complete.")

## Oefeningen

* Voeg een extra Dense-laag toe na de RNN-laag. Experimenteer met het aantal neuronen in deze laag en analyseer hoe de prestaties veranderen.
* Pas het model aan om in plaats van een SimpleRNN-laag een LSTM of GRU-laag te gebruiken. Vergelijk de prestaties van de drie typen recurrente netwerken.

In [ ]:
# Oefening 1

In [ ]:
# Oefening 2

**Oefening 3**

Volg de tutorial op de volgende link: https://www.tensorflow.org/text/tutorials/text_generation
Werk hieronder het gelijkaardige probleem uit maar maak het door gebruik te maken van pytorch in plaats van tensorflow voor het model op te bouwen.
In deze tutorial wordt er tekst gegenereerd die lijkt op tekst geschreven door shakespeare.
Let op dat dit een vereenvoudigde versie is waarbij karakter per karakter wordt gegenereerd en niet woord per woord. Er is dus geen garantie dat er echte woorden gemaakt worden.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset, Subset
import random
from urllib.request import urlretrieve
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Download Shakespeare-tekstbestand
url = "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
path_to_file = Path("shakespeare.txt")

if not path_to_file.exists():
    urlretrieve(url, path_to_file)

text = path_to_file.read_text(encoding="utf-8")

print(f'Length of text: {len(text)} characters')
print(text[:250])

# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

# Character to index mapping
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}


# TODO: Encodeer elk karakter in tekst naar een nummer, uitkomst is een list ipv een string

# TODO: Maak een dataset aan waarbij de tekst (uit voorgaande todo) omzet naar een reeks sequenties
# input 100 aaneensluitende karakters, output is het karakter erop volgende

# TODO: indien nodig maak een subset tot 10 of 1% van de dataset

# Check a single example
sample_x, sample_y = dataset[0]
print("Input (x):", sample_x)
print("Target (y):", sample_y)
print("Decoded Input:", ''.join(idx_to_char[idx] for idx in sample_x.numpy()))
print("Decoded Target:", idx_to_char[sample_y.item()])
print('Rows', len(dataset))

In [ ]:
# TODO: Maak een rnn model bestaande uit een embedding layer, gru layer en linear layer
# Maak het mogelijk om aan de forward funtie een parameter toe te voegen om ook de hidden state terug te geven en om de hidden state mee te geven voor de gru laag
# 
vocab_size = len(idx_to_char)
print(vocab_size)
embedding_dim = 50
rnn_units = 60

In [ ]:
# test 1 sample om door het model te sturen
# kijk of je dimensies correct aan elkaar gekoppeld zijn

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import os
import math

batch_size = 64
seq_length = 100
epochs = 5
vocab_size = len(vocab)
embedding_dim = 50
rnn_units = 60

shakespeare = ShakespeareModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

# TODO: train het rnn model

In [ ]:
import torch.nn.functional as F

def generate_text(model, start_string, char_to_idx, idx_to_char, vocab_size, generation_length=100, temperature=1.0):
    model.eval()  # Set model to evaluation mode
    
    # Convert start_string to indices
    input_indices = torch.tensor([char_to_idx[char] for char in start_string], dtype=torch.long).unsqueeze(0)
    
    generated_text = start_string
    states = None  # Initial state (None means it will be initialized automatically)
    
    for _ in range(generation_length):
        # Genereer opeenvolgend nieuwe tokens
        pass
    
    return generated_text


In [ ]:
# Example start string and generation parameters
start_string = "ROMEO: "
generation_length = 200
temperature = 0.8

# Generate text
generated_text = generate_text(
    model=shakespeare,
    start_string=start_string,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    vocab_size=vocab_size,
    generation_length=generation_length,
    temperature=temperature
)

print("Generated Text:")
print(generated_text)
